# Library 01 – Loading the Cricsheet archive and filtering matches

Functions for **Stage 1** (source manifest) and **Stage 2** (match filters).

* The zip archive is the untouched source. We extract it once into `data/raw/england_male_json/`.
* Every JSON file gets a row in the manifest, with its size and SHA-256 checksum.
* Every file is then either **included** or **excluded with one written reason**. Nothing is skipped silently.

Needs: `lib_00_settings`.

In [ ]:
def file_checksum(file_path: Path, block_size: int = 1 << 20) -> str:
    """Return the SHA-256 checksum of a file, read in blocks so large files use little memory."""
    checksum = hashlib.sha256()
    with file_path.open("rb") as open_file:
        while block := open_file.read(block_size):
            checksum.update(block)
    return checksum.hexdigest()


def extract_archive_if_needed(archive_path: Path = RAW_ARCHIVE_PATH, match_folder: Path = RAW_MATCH_FOLDER) -> Path:
    """Unzip the Cricsheet archive unless the match files are already there. Returns the match folder."""
    already_extracted = match_folder.exists() and any(match_folder.glob("*.json"))
    if not already_extracted:
        if not archive_path.exists():
            raise FileNotFoundError(f"{archive_path} is missing. Download it from {ARCHIVE_SOURCE_URL} into data/raw/.")
        match_folder.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(match_folder)
    return match_folder


def list_match_files(match_folder: Path = RAW_MATCH_FOLDER) -> list[Path]:
    """All match JSON files, sorted by name so every run sees them in the same order."""
    return sorted(match_folder.glob("*.json"))


def load_match_file(match_path: Path) -> dict[str, Any]:
    """Read one Cricsheet match file into a Python dictionary."""
    with match_path.open(encoding="utf-8") as open_file:
        return json.load(open_file)

## Source manifest

One row per file. A file that cannot be read stays in the manifest with `readable = False`.

In [ ]:
def build_source_manifest(match_folder: Path = RAW_MATCH_FOLDER) -> pd.DataFrame:
    """Describe every source file: name, size, checksum and Cricsheet schema version."""
    manifest_rows = []
    for match_path in list_match_files(match_folder):
        manifest_row = {
            "filename": match_path.name,
            "bytes": match_path.stat().st_size,
            "sha256": file_checksum(match_path),
            "readable": True,
            "data_version": "",
            "created": "",
            "revision": -1,
        }
        try:
            file_meta = load_match_file(match_path).get("meta", {})
            manifest_row["data_version"] = str(file_meta.get("data_version", ""))
            manifest_row["created"] = str(file_meta.get("created", ""))
            manifest_row["revision"] = int(file_meta.get("revision", -1))
        except (json.JSONDecodeError, OSError, ValueError):
            manifest_row["readable"] = False
        manifest_rows.append(manifest_row)
    return pd.DataFrame(manifest_rows)

## Match metadata and filters

`describe_match` flattens the `info` block of a match into one row. `find_exclusion_reason` applies the
filters **in a fixed order** and returns the first one that fails, or an empty string if the match qualifies.

In [ ]:
def to_date(value: Any) -> date:
    """Turn a Cricsheet date (an ISO text such as '2022-06-01') into a ``date``."""
    return value if isinstance(value, date) else date.fromisoformat(str(value))


def describe_match(match_data: dict[str, Any], filename: str) -> dict[str, Any]:
    """Flatten the match information we need into one record."""
    match_info = match_data["info"]
    match_days = [to_date(day) for day in match_info.get("dates", [])]
    team_names = list(match_info.get("teams", []))
    match_outcome = match_info.get("outcome", {})
    match_event = match_info.get("event", {}) or {}
    all_innings = match_data.get("innings", [])
    return {
        "match_id": filename.rsplit(".", 1)[0],
        "filename": filename,
        "start_date": min(match_days) if match_days else None,
        "end_date": max(match_days) if match_days else None,
        "gender": match_info.get("gender"),
        "team_type": match_info.get("team_type"),
        "match_type": match_info.get("match_type"),
        "format": CRICSHEET_TO_FORMAT.get(match_info.get("match_type", ""), ""),
        "team_1": team_names[0] if len(team_names) > 0 else "",
        "team_2": team_names[1] if len(team_names) > 1 else "",
        "opponent": next((team for team in team_names if team != TEAM_NAME), ""),
        "event": match_event.get("name", ""),
        "venue": match_info.get("venue", ""),
        "outcome_winner": match_outcome.get("winner", ""),
        "outcome_result": match_outcome.get("result", ""),
        "outcome_method": match_outcome.get("method", ""),
        "n_innings_total": len(all_innings),
        "n_innings_england": sum(1 for innings in all_innings if innings.get("team") == TEAM_NAME),
    }


def find_exclusion_reason(match_record: dict[str, Any]) -> str:
    """Return '' if the match qualifies, otherwise the name of the first filter it fails."""
    if match_record["gender"] != TEAM_GENDER:
        return "gender_not_male"
    if match_record["team_type"] != TEAM_TYPE:
        return "not_international"
    if TEAM_NAME not in (match_record["team_1"], match_record["team_2"]):
        return "england_not_playing"
    if not match_record["format"]:
        return "format_not_target"
    if match_record["start_date"] is None:
        return "missing_date"
    if match_record["start_date"] < STUDY_START_DATE:
        return "before_window"
    if match_record["start_date"] > STUDY_END_DATE:
        return "after_window"
    return ""


def audit_match_files(match_paths: Iterable[Path]) -> pd.DataFrame:
    """Describe every file and mark it as included or excluded (with the reason)."""
    audit_rows = []
    for match_path in match_paths:
        try:
            match_record = describe_match(load_match_file(match_path), match_path.name)
            exclusion_reason = find_exclusion_reason(match_record)
        except Exception as error:  # a broken file is recorded, never skipped silently
            match_record = {"match_id": match_path.stem, "filename": match_path.name}
            exclusion_reason = f"unreadable:{type(error).__name__}"
        match_record["included"] = exclusion_reason == ""
        match_record["exclusion_reason"] = exclusion_reason
        audit_rows.append(match_record)
    return pd.DataFrame(audit_rows)